In [3]:
import pandas as pd
import numpy as np
import plotly.express as px 
from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_selector,ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer

from xgboost import XGBRegressor

import joblib

from pycaret.regression import *

import warnings
warnings.filterwarnings("ignore")

In [4]:
df = pd.read_csv("survey_results_public.csv")
df.head()

,ResponseId,MainBranch,Employment,RemoteWork,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,LearnCodeCoursesCert,YearsCode,...,TimeSearching,TimeAnswering,Onboarding,ProfessionalTech,TrueFalse_1,TrueFalse_2,TrueFalse_3,SurveyLength,SurveyEase,ConvertedCompYearly
0,1,None of these,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,I am a developer by profession,"Employed, full-time",Fully remote,Hobby;Contribute to open-source projects,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Too long,Difficult,NaN
2,3,"I am not primarily a developer, but I write co...","Employed, full-time","Hybrid (some remote, some in-person)",Hobby,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Books / Physical media;Friend or family member...,Technical documentation;Blogs;Programming Game...,NaN,14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Appropriate in length,Neither easy nor difficult,40205.0
3,4,I am a developer by profession,"Employed, full-time",Fully remote,I don’t code outside of work,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Books / Physical media;School (i.e., Universit...",NaN,NaN,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Appropriate in length,Easy,215232.0
4,5,I am a developer by profession,"Employed, full-time","Hybrid (some remote, some in-person)",Hobby,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Other online resources (e.g., videos, blogs, f...",Technical documentation;Blogs;Stack Overflow;O...,NaN,8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Too long,Easy,NaN


In [5]:
df.shape

(73268, 79)

In [6]:
df.columns

Index(['ResponseId', 'MainBranch', 'Employment', 'RemoteWork',
       'CodingActivities', 'EdLevel', 'LearnCode', 'LearnCodeOnline',
       'LearnCodeCoursesCert', 'YearsCode', 'YearsCodePro', 'DevType',
       'OrgSize', 'PurchaseInfluence', 'BuyNewTool', 'Country', 'Currency',
       'CompTotal', 'CompFreq', 'LanguageHaveWorkedWith',
       'LanguageWantToWorkWith', 'DatabaseHaveWorkedWith',
       'DatabaseWantToWorkWith', 'PlatformHaveWorkedWith',
       'PlatformWantToWorkWith', 'WebframeHaveWorkedWith',
       'WebframeWantToWorkWith', 'MiscTechHaveWorkedWith',
       'MiscTechWantToWorkWith', 'ToolsTechHaveWorkedWith',
       'ToolsTechWantToWorkWith', 'NEWCollabToolsHaveWorkedWith',
       'NEWCollabToolsWantToWorkWith', 'OpSysProfessional use',
       'OpSysPersonal use', 'VersionControlSystem', 'VCInteraction',
       'VCHostingPersonal use', 'VCHostingProfessional use',
       'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith',
       'OfficeStackSyncHaveWork

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73268 entries, 0 to 73267
Data columns (total 79 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   ResponseId                      73268 non-null  int64  
 1   MainBranch                      73268 non-null  object 
 2   Employment                      71709 non-null  object 
 3   RemoteWork                      58958 non-null  object 
 4   CodingActivities                58899 non-null  object 
 5   EdLevel                         71571 non-null  object 
 6   LearnCode                       71580 non-null  object 
 7   LearnCodeOnline                 50685 non-null  object 
 8   LearnCodeCoursesCert            29389 non-null  object 
 9   YearsCode                       71331 non-null  object 
 10  YearsCodePro                    51833 non-null  object 
 11  DevType                         61302 non-null  object 
 12  OrgSize                         

In [8]:
df.Currency.value_counts()

Currency
EUR European Euro              12634
USD\tUnited States dollar      12346
INR\tIndian rupee               4229
GBP\tPound sterling             3318
CAD\tCanadian dollar            1847
                               ...  
BND\tBrunei dollar                 1
PGK\tPapua New Guinean kina        1
SHP\tSaint Helena pound            1
GIP\tGibraltar pound               1
TOP\tTongan pa’anga                1
Name: count, Length: 142, dtype: int64

In [9]:
df.isnull().sum()

ResponseId                 0
MainBranch                 0
Employment              1559
RemoteWork             14310
CodingActivities       14369
                       ...  
TrueFalse_2            37553
TrueFalse_3            37519
SurveyLength            2824
SurveyEase              2760
ConvertedCompYearly    35197
Length: 79, dtype: int64

In [10]:
def plot_bar(df, column, line=""):
    if line=="":
        line = df[column].value_counts().keys()[:10]
    data = df[column].value_counts()[0:10]

    fig = px.bar(x=line, y=data)

    fig.show()

plot_bar(df,"Country")

In [11]:
plot_bar(df,"EdLevel")

In [12]:
px.pie(data_frame=df, names="OrgSize")

In [13]:
df1 = df[["Country", "EdLevel", "YearsCodePro", "ConvertedCompYearly"]]
df1.head()

,Country,EdLevel,YearsCodePro,ConvertedCompYearly
0,NaN,NaN,NaN,NaN
1,Canada,NaN,NaN,NaN
2,United Kingdom of Great Britain and Northern I...,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",5,40205.0
3,Israel,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",17,215232.0
4,United States of America,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",3,NaN


In [14]:
df1 = df1.rename({"ConvertedCompYearly":"Salary"}, axis=1)
df1.sample(5)

,Country,EdLevel,YearsCodePro,Salary
61719,Russian Federation,"Secondary school (e.g. American high school, G...",NaN,NaN
15462,Canada,"Secondary school (e.g. American high school, G...",NaN,NaN
52091,Kenya,Some college/university study without earning ...,NaN,NaN
30619,United States of America,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",2,130000.0
20515,India,Some college/university study without earning ...,NaN,NaN


In [15]:
df1.isnull().sum()

Country          1497
EdLevel          1697
YearsCodePro    21435
Salary          35197
dtype: int64

In [16]:
df1 = df1.dropna()
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 37923 entries, 2 to 73121
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Country       37923 non-null  object 
 1   EdLevel       37923 non-null  object 
 2   YearsCodePro  37923 non-null  object 
 3   Salary        37923 non-null  float64
dtypes: float64(1), object(3)
memory usage: 1.4+ MB


In [17]:
df1 = df1[df1["Salary"] <= 200000]
df1 = df1[df1["Salary"] >= 10000]
df1 = df1[df1["Country"] != "Other"]

In [18]:
df1.YearsCodePro.value_counts().keys()


Index(['5', '3', '4', '2', '10', '6', '7', '8', '1', '12', '15', '9',
       'Less than 1 year', '11', '20', '14', '13', '16', '22', '18', '25',
       '17', '30', '23', '24', '21', '19', '26', '27', '35', '28', '32', '40',
       '29', '31', '36', '34', '37', '33', '38', '42', '41', '39', '43', '45',
       '44', 'More than 50 years', '46', '50', '48', '47', '49'],
      dtype='object', name='YearsCodePro')

In [19]:
def clean_years(x):
    if x == "Less than 1 year":
        return 0.5
    if x == "More than 50 years":
        return 50
    return float(x)

df1.YearsCodePro = df1.YearsCodePro.apply(clean_years)

In [20]:
# its bad
# imputer = SimpleImputer(missing_values=np.nan, strategy="mean")
# df1[["Salary", "YearsCodePro"]] = imputer.fit_transform(df1[["Salary", "YearsCodePro"]])

In [21]:
df1.Country.unique()

array(['United Kingdom of Great Britain and Northern Ireland',
       'Netherlands', 'United States of America', 'Czech Republic',
       'Austria', 'Italy', 'Canada', 'Germany', 'Poland', 'Israel',
       'Norway', 'Taiwan', 'France', 'Brazil', 'Uruguay', 'Sweden',
       'Spain', 'Turkey', 'Romania', 'India', 'Belgium', 'Bulgaria',
       'Ireland', 'Russian Federation', 'Saudi Arabia', 'Mexico',
       'Switzerland', 'Latvia', 'South Africa', 'Thailand', 'China',
       'Montenegro', 'Finland', 'Slovakia', 'Denmark', 'Australia',
       'Greece', 'Portugal', 'Argentina', 'Hungary', 'Ukraine',
       'Maldives', 'Hong Kong (S.A.R.)', 'Serbia', 'Singapore', 'Egypt',
       'Croatia', 'Indonesia', 'Armenia', 'Lithuania',
       'Iran, Islamic Republic of...', 'Belarus', 'Bangladesh',
       'Costa Rica', 'Pakistan', 'Mauritius', 'Estonia', 'Kazakhstan',
       'Morocco', 'Philippines', 'Chile', 'Slovenia', 'New Zealand',
       'Ecuador', 'Cyprus', 'Japan', 'Peru', 'Afghanistan', 'Nica

In [22]:
def short_category(catg):
    cut = 200
    catg_map = {}
    for i in range(len(catg)):
        if catg.values[i] >= cut:
            catg_map[catg.index[i]] = catg.index[i]
        else:
            catg_map[catg.index[i]] = "Other"
    return catg_map


In [23]:
country_map = short_category(df1.Country.value_counts())
df1.Country = df.Country.map(country_map)
df1.Country.value_counts()

Country
United States of America                                6666
Other                                                   4060
Germany                                                 2695
United Kingdom of Great Britain and Northern Ireland    2262
India                                                   1411
Canada                                                  1327
France                                                  1289
Brazil                                                  1105
Poland                                                   940
Spain                                                    877
Netherlands                                              838
Australia                                                748
Italy                                                    696
Sweden                                                   636
Russian Federation                                       480
Switzerland                                              459
Austria         

In [24]:
df1.EdLevel.value_counts()

EdLevel
Bachelor’s degree (B.A., B.S., B.Eng., etc.)                                          14618
Master’s degree (M.A., M.S., M.Eng., MBA, etc.)                                        8500
Some college/university study without earning a degree                                 3757
Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)     1494
Other doctoral degree (Ph.D., Ed.D., etc.)                                             1157
Associate degree (A.A., A.S., etc.)                                                    1081
Professional degree (JD, MD, etc.)                                                      502
Something else                                                                          301
Primary/elementary school                                                               184
Name: count, dtype: int64

In [25]:
def clean_ed(x):
    if "Bachelor’s degree" in x:
        return "Bachelor’s degree"
    if "Master’s degree" in x:
        return "Master’s degree"
    if "Professional degree (JD, MD, etc.)" in x  or "Other doctoral degree (Ph.D., Ed.D., etc.)" in x:
        return "Post grad"
    return "less than Bachelors"

df1["EdLevel"]= df1["EdLevel"].apply(clean_ed)

In [26]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31594 entries, 2 to 73121
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Country       31594 non-null  object 
 1   EdLevel       31594 non-null  object 
 2   YearsCodePro  31594 non-null  float64
 3   Salary        31594 non-null  float64
dtypes: float64(2), object(2)
memory usage: 1.2+ MB


In [27]:
setup(df1, target="Salary", session_id=23)

,Description,Value
0,Session id,23
1,Target,Salary
2,Target type,Regression
3,Original data shape,"(31594, 4)"
4,Transformed data shape,"(31594, 7)"
5,Transformed train set shape,"(22115, 7)"
6,Transformed test set shape,"(9479, 7)"
7,Numeric features,1
8,Categorical features,2
9,Preprocess,True


In [28]:
models()

,Name,Reference,Turbo
ID,,,
lr,Linear Regression,sklearn.linear_model._base.LinearRegression,True
lasso,Lasso Regression,sklearn.linear_model._coordinate_descent.Lasso,True
ridge,Ridge Regression,sklearn.linear_model._ridge.Ridge,True
en,Elastic Net,sklearn.linear_model._coordinate_descent.Elast...,True
lar,Least Angle Regression,sklearn.linear_model._least_angle.Lars,True
llar,Lasso Least Angle Regression,sklearn.linear_model._least_angle.LassoLars,True
omp,Orthogonal Matching Pursuit,sklearn.linear_model._omp.OrthogonalMatchingPu...,True
br,Bayesian Ridge,sklearn.linear_model._bayes.BayesianRidge,True
ard,Automatic Relevance Determination,sklearn.linear_model._bayes.ARDRegression,False


In [29]:
compare_models()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
gbr,Gradient Boosting Regressor,22819.6553,959531259.5095,30970.4189,0.5561,0.4721,0.4387,0.0860
lightgbm,Light Gradient Boosting Machine,22840.9724,964720005.6825,31053.4220,0.5537,0.4721,0.4371,0.1010
catboost,CatBoost Regressor,22877.8808,970221856.5680,31142.4432,0.5511,0.4734,0.4378,0.9800
xgboost,Extreme Gradient Boosting,23258.7922,1002852064.0000,31662.3508,0.5360,0.4814,0.4431,0.1920
lasso,Lasso Regression,23985.7471,1033610210.3916,32144.4952,0.5218,0.4912,0.4666,0.2680
ridge,Ridge Regression,23985.7483,1033609942.1331,32144.4906,0.5218,0.4912,0.4666,0.0150
lar,Least Angle Regression,23985.7497,1033610066.5189,32144.4925,0.5218,0.4912,0.4666,0.0150
llar,Lasso Least Angle Regression,23985.7471,1033610210.4750,32144.4952,0.5218,0.4912,0.4666,0.0240
br,Bayesian Ridge,23985.7060,1033607740.8690,32144.4589,0.5218,0.4912,0.4666,0.0160
lr,Linear Regression,23985.7497,1033610066.5189,32144.4925,0.5218,0.4912,0.4666,0.3900


GradientBoostingRegressor(random_state=23)

In [30]:
X = df1.drop("Salary", axis=1)
y = df1.Salary

In [31]:
numeric_pipe = Pipeline([("Scaler", StandardScaler())])
cat_pipe = Pipeline([("OneHotEncoder" , OneHotEncoder(handle_unknown="ignore"))])

transform = ColumnTransformer([("numeric", numeric_pipe, ["YearsCodePro"]), ("cat", cat_pipe, ["EdLevel", "Country"])])

In [32]:
x_train, x_test, y_train,y_test = train_test_split(X,y,test_size=0.2, random_state=23)

LinearRegression

In [33]:
mlp1 = Pipeline([("Transform", transform), ("lr", LinearRegression())])
mlp1.fit(x_train,y_train)

Pipeline(steps=[('Transform',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('Scaler',
                                                                   StandardScaler())]),
                                                  ['YearsCodePro']),
                                                 ('cat',
                                                  Pipeline(steps=[('OneHotEncoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['EdLevel', 'Country'])])),
                ('lr', LinearRegression())])

In [34]:
y_hat = mlp1.predict(x_test)
y_hat

array([ 39193.77048342,  48394.21312532,  70306.23436623, ...,
        84351.94080359, 107926.81397183,  39174.09652323])

In [35]:
error = np.sqrt(mean_squared_error(y_test,y_hat))
print("${:,.02f}".format(error))

$32,058.81


DecisionTreeRegressor


In [36]:
mlp2 = Pipeline([("Transform", transform), ("dr", DecisionTreeRegressor())])
mlp2.fit(x_train,y_train)

Pipeline(steps=[('Transform',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('Scaler',
                                                                   StandardScaler())]),
                                                  ['YearsCodePro']),
                                                 ('cat',
                                                  Pipeline(steps=[('OneHotEncoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['EdLevel', 'Country'])])),
                ('dr', DecisionTreeRegressor())])

In [37]:
y_hat = mlp2.predict(x_test)
error = np.sqrt(mean_squared_error(y_test,y_hat))
print("${:,.02f}".format(error))

$33,592.74


RandomForestRegressor

In [38]:
mlp3 = Pipeline([("Transform", transform), ("rfr", RandomForestRegressor())])
mlp3.fit(x_train,y_train)

Pipeline(steps=[('Transform',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('Scaler',
                                                                   StandardScaler())]),
                                                  ['YearsCodePro']),
                                                 ('cat',
                                                  Pipeline(steps=[('OneHotEncoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['EdLevel', 'Country'])])),
                ('rfr', RandomForestRegressor())])

In [39]:
y_hat = mlp3.predict(x_test)
error = np.sqrt(mean_squared_error(y_test,y_hat))
print("${:,.02f}".format(error))

$32,505.63


XGBRegressor

In [40]:
mlp4 = Pipeline([("Transform", transform), ("xgbr", XGBRegressor())])
mlp4.fit(x_train,y_train)

Pipeline(steps=[('Transform',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('Scaler',
                                                                   StandardScaler())]),
                                                  ['YearsCodePro']),
                                                 ('cat',
                                                  Pipeline(steps=[('OneHotEncoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['EdLevel', 'Country'])])),
                ('xgbr',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=N...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=None,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=None, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [41]:
y_hat = mlp4.predict(x_test)
error = np.sqrt(mean_squared_error(y_test,y_hat))
print("${:,.02f}".format(error))

$31,416.02


GradientBoostingRegressor

In [42]:
mlp5 = Pipeline([("Transform", transform), ("gbr", GradientBoostingRegressor())])
mlp5.fit(x_train,y_train)

Pipeline(steps=[('Transform',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('Scaler',
                                                                   StandardScaler())]),
                                                  ['YearsCodePro']),
                                                 ('cat',
                                                  Pipeline(steps=[('OneHotEncoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['EdLevel', 'Country'])])),
                ('gbr', GradientBoostingRegressor())])

In [43]:
y_hat = mlp5.predict(x_test)
eror = np.sqrt(mean_squared_error(y_test,y_hat))
print("${:,.02f}".format(error))

$31,416.02


In [44]:
joblib.dump(mlp5, "gbr_n.joblib")

['gbr_n.joblib']